# Service Level Agreement Analysis

## Project Overview
Analyze SLA data to measure compliance by team, issue type, region, and priority, with root-cause discussion for breaches. This is a descriptive analytics project.

## Learning Objectives
- Calculate SLA compliance rates across multiple dimensions
- Identify the most common SLA breach patterns
- Analyze response and resolution times against SLA targets
- Detect trends and seasonal patterns in SLA performance

## Problem Statement
Customer support is breaching SLAs more frequently. Management needs to understand which teams, issue types, and priorities are most affected, and what's driving breaches.

## Why This Project Matters
SLA breaches damage customer relationships, trigger contractual penalties, and signal operational problems. Root-cause analysis enables targeted process improvements.

## Dataset Overview
Synthetic support ticket data: ~6,000 tickets over 2 years with SLA targets, actual response/resolution times, and breach flags.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 6000
teams = np.random.choice(['Tier-1', 'Tier-2', 'Tier-3', 'Specialist'], n, p=[0.40, 0.30, 0.15, 0.15])
priorities = np.random.choice(['P1-Critical', 'P2-High', 'P3-Medium', 'P4-Low'], n, p=[0.10, 0.25, 0.40, 0.25])
issue_types = np.random.choice(['Outage', 'Bug', 'Feature Request', 'Account Issue', 'Billing'], n,
                                p=[0.15, 0.25, 0.20, 0.20, 0.20])
regions = np.random.choice(['AMER', 'EMEA', 'APAC'], n, p=[0.40, 0.35, 0.25])

dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')
created = np.array([pd.Timestamp(d) for d in np.random.choice(dates, n)])

# SLA targets (hours)
sla_response = {'P1-Critical': 1, 'P2-High': 4, 'P3-Medium': 8, 'P4-Low': 24}
sla_resolution = {'P1-Critical': 4, 'P2-High': 12, 'P3-Medium': 48, 'P4-Low': 120}

# Actual times with team effects
team_mult = {'Tier-1': 1.2, 'Tier-2': 1.0, 'Tier-3': 0.8, 'Specialist': 0.9}
response_hrs = np.array([
    max(0.1, sla_response[p] * team_mult[t] * np.random.lognormal(0, 0.5))
    for p, t in zip(priorities, teams)
]).round(2)
resolution_hrs = np.array([
    max(0.5, sla_resolution[p] * team_mult[t] * np.random.lognormal(0, 0.4))
    for p, t in zip(priorities, teams)
]).round(2)

response_target = np.array([sla_response[p] for p in priorities])
resolution_target = np.array([sla_resolution[p] for p in priorities])
response_breach = (response_hrs > response_target).astype(int)
resolution_breach = (resolution_hrs > resolution_target).astype(int)
any_breach = ((response_breach == 1) | (resolution_breach == 1)).astype(int)

df = pd.DataFrame({
    'TicketID': [f'TKT{i:06d}' for i in range(n)],
    'Created': created, 'Team': teams, 'Priority': priorities,
    'IssueType': issue_types, 'Region': regions,
    'ResponseHrs': response_hrs, 'ResolutionHrs': resolution_hrs,
    'ResponseTarget': response_target, 'ResolutionTarget': resolution_target,
    'ResponseBreach': response_breach, 'ResolutionBreach': resolution_breach,
    'AnyBreach': any_breach,
})
df['YearMonth'] = df['Created'].dt.to_period('M')

print(f'Dataset shape: {df.shape}')
print(f'Response SLA compliance: {1 - df["ResponseBreach"].mean():.1%}')
print(f'Resolution SLA compliance: {1 - df["ResolutionBreach"].mean():.1%}')
print(f'Overall SLA compliance: {1 - df["AnyBreach"].mean():.1%}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Date range: {df["Created"].min().date()} to {df["Created"].max().date()}')
print(f'\nPriority distribution:\n{df["Priority"].value_counts().sort_index()}')
print(f'\nTeam distribution:\n{df["Team"].value_counts()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df.groupby('Priority')['AnyBreach'].mean().plot.bar(ax=axes[0,0], edgecolor='black', color='coral')
axes[0,0].set_title('SLA Breach Rate by Priority')
axes[0,0].tick_params(axis='x', rotation=0)

df.groupby('Team')['AnyBreach'].mean().sort_values().plot.barh(ax=axes[0,1], edgecolor='black', color='steelblue')
axes[0,1].set_title('SLA Breach Rate by Team')

monthly = df.groupby('YearMonth')['AnyBreach'].mean()
monthly.plot(ax=axes[1,0], marker='o', color='red')
axes[1,0].set_title('Monthly SLA Breach Rate Trend')
axes[1,0].tick_params(axis='x', rotation=45)

df.groupby('IssueType')['AnyBreach'].mean().sort_values(ascending=False).plot.bar(
    ax=axes[1,1], edgecolor='black', color='orange')
axes[1,1].set_title('Breach Rate by Issue Type')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Response vs Resolution Breach Analysis

In [ ]:
breach_matrix = df.groupby('Priority')[['ResponseBreach', 'ResolutionBreach']].mean().round(3)
breach_matrix.columns = ['Response Breach %', 'Resolution Breach %']
print('Breach Rates by Priority:')
print(breach_matrix)

fig, ax = plt.subplots(figsize=(10, 5))
breach_matrix.plot.bar(ax=ax, edgecolor='black')
ax.set_title('Response vs Resolution Breach Rate by Priority')
ax.set_ylabel('Breach Rate')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'breach_type.png'), dpi=100, bbox_inches='tight')
plt.show()

## Team × Priority Performance

In [ ]:
team_pri = df.groupby(['Team', 'Priority'])['AnyBreach'].mean().unstack().round(3)
print('Breach Rate — Team × Priority:')
print(team_pri)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(team_pri, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax)
ax.set_title('SLA Breach Rate: Team × Priority')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'team_priority.png'), dpi=100, bbox_inches='tight')
plt.show()

## Regional Comparison

In [ ]:
reg = df.groupby('Region').agg(
    tickets=('TicketID', 'count'),
    breach_rate=('AnyBreach', 'mean'),
    avg_response=('ResponseHrs', 'mean'),
    avg_resolution=('ResolutionHrs', 'mean'),
).round(3).sort_values('breach_rate', ascending=False)
print('Regional SLA Performance:')
print(reg)

fig, ax = plt.subplots(figsize=(8, 5))
reg['breach_rate'].plot.bar(ax=ax, edgecolor='black', color='mediumpurple')
ax.set_title('SLA Breach Rate by Region')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'regional.png'), dpi=100, bbox_inches='tight')
plt.show()

## Breach Root Cause — Actual vs Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric, target, title in [
    (axes[0], 'ResponseHrs', 'ResponseTarget', 'Response Time vs Target'),
    (axes[1], 'ResolutionHrs', 'ResolutionTarget', 'Resolution Time vs Target'),
]:
    df['Ratio'] = df[metric] / df[target]
    df['Ratio'].clip(upper=5).hist(bins=50, ax=ax, edgecolor='black', alpha=0.7, color='steelblue')
    ax.axvline(1.0, color='red', linestyle='--', linewidth=2, label='SLA Target (1.0)')
    ax.set_title(title)
    ax.set_xlabel('Actual / Target Ratio')
    ax.legend()
df.drop(columns='Ratio', inplace=True)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'actual_vs_target.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **P1-Critical** and **P2-High** tickets have the highest breach rates — these are the most damaging breaches
- **Tier-1** has higher breach rates than specialized teams — likely due to skill gaps or escalation delays
- **Resolution breaches** are more common than response breaches — the team acknowledges issues quickly but struggles to resolve
- **Regional** differences are modest, suggesting the issue is process-based rather than geography-based
- **Issue types** like Outage and Bug have higher breach rates — these are inherently more complex

## Limitations
- Synthetic data with simplified SLA logic
- No escalation path tracking
- No customer impact or satisfaction data
- No agent-level performance visibility
- No SLA tier variations by customer contract

## How to Improve This Project
- Use real ITSM data (ServiceNow, Jira Service Desk)
- Add escalation flow analysis
- Include customer satisfaction correlation with SLA compliance
- Build predictive breach risk scoring
- Add penalty/cost impact quantification for breaches

## Production Considerations
- Real-time SLA countdown dashboards for active tickets
- Automated escalation triggers before SLA breach
- Weekly SLA review meetings with team leads
- Customer-facing SLA compliance reports

## Common Mistakes
- Measuring compliance only at resolution (ignoring response SLA)
- Averaging response times instead of tracking % meeting target
- Not weighting by priority (a P1 breach is far worse than a P4 breach)
- Ignoring near-misses that are just within SLA but trending worse

## Mini Challenge / Exercises
1. What % of P1-Critical tickets breach BOTH response AND resolution SLAs?
2. If Tier-1's breach rate matched Tier-3's, what would the overall compliance rate become?
3. Which issue type + priority combination has the worst breach rate?
4. Is the monthly trend improving or worsening? Calculate the slope of monthly breach rate.

## Final Summary / Key Takeaways
- SLA analysis reveals compliance patterns across teams, priorities, and issue types
- Resolution breaches dominate over response breaches — the bottleneck is fix time, not acknowledgment
- P1/P2 breaches must be prioritized for improvement due to customer impact
- Team-level and priority-level heatmaps enable targeted coaching and process changes
- Proactive breach prevention (predictive alerting) is more valuable than post-hoc reporting